# 00 · Kiểm tra chất lượng dữ liệu OULAD


In [1]:
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SEED = 42                      
np.random.seed(SEED)
CUTOFF_DAY = 28                

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = Path(os.environ.get("OULAD_RAW_DIR", ROOT / "data" / "raw"))
RESULTS_DIR = Path(os.environ.get("OULAD_RESULTS_DIR", ROOT / "results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

KEYS = ["id_student", "code_module", "code_presentation"]
TABLES = ["courses", "assessments", "vle", "studentInfo",
          "studentRegistration", "studentAssessment", "studentVle"]

## 1. Đọc 7 bảng

`studentVle` có hơn 10 triệu dòng, máy nhóm chỉ có 8 GiB RAM, nên ép kiểu số nguyên nhỏ và `category` cho cột mã. Ký tự `?` và ô rỗng đều được coi là thiếu (`na_values`); số ô `?` thô được đếm riêng để ghi vào báo cáo.

In [2]:
missing_files = [t for t in TABLES if not (RAW_DIR / f"{t}.csv").exists()]
if missing_files:
    raise FileNotFoundError(
        f"Thiếu file trong {RAW_DIR}: {[t + '.csv' for t in missing_files]}. "
        "Hãy đặt đủ 7 CSV gốc OULAD vào data/raw/ hoặc đặt biến OULAD_RAW_DIR."
    )

NA = ["?", ""]
COMPACT = {
    "studentVle": {"code_module": "category", "code_presentation": "category",
                   "id_student": "int32", "id_site": "int32",
                   "date": "int16", "sum_click": "int16"},
    "studentInfo": {"id_student": "int32"},
    "studentRegistration": {"id_student": "int32"},
    "studentAssessment": {"id_student": "int32", "id_assessment": "int32"},
}

def read_table(name):
    path = RAW_DIR / f"{name}.csv"
    try:
        return pd.read_csv(path, na_values=NA, dtype=COMPACT.get(name))
    except (ValueError, OverflowError) as e:      # có NaN trong cột int, v.v. -> đọc mặc định
        print(f"[{name}] không ép kiểu gọn được ({e}); đọc mặc định.")
        return pd.read_csv(path, na_values=NA)

def count_question_marks(name):
    # đếm ô '?' thô để biết dữ liệu dùng '?' hay ô rỗng cho giá trị thiếu
    if name == "studentVle":
        return 0
    raw = pd.read_csv(RAW_DIR / f"{name}.csv", dtype=str, keep_default_na=False)
    return int((raw == "?").sum().sum())

t0 = time.time()
df = {}
for name in TABLES:
    t = time.time()
    df[name] = read_table(name)
    print(f"{name:20s} {len(df[name]):>11,d} dòng × {df[name].shape[1]:>2d} cột   ({time.time()-t:5.1f}s)")
print(f"Tổng thời gian đọc: {time.time()-t0:.1f}s")

qmarks = {n: count_question_marks(n) for n in TABLES}
print("Số ô '?' thô theo bảng:", {k: v for k, v in qmarks.items() if v})

courses                       22 dòng ×  3 cột   (  0.0s)
assessments                  206 dòng ×  6 cột   (  0.0s)
vle                        6,364 dòng ×  6 cột   (  0.0s)
studentInfo               32,593 dòng × 12 cột   (  0.1s)
studentRegistration       32,593 dòng ×  5 cột   (  0.0s)
studentAssessment        173,912 dòng ×  5 cột   (  0.2s)
studentVle            10,655,280 dòng ×  6 cột   (  8.7s)
Tổng thời gian đọc: 9.0s
Số ô '?' thô theo bảng: {'assessments': 11, 'vle': 10486, 'studentInfo': 1111, 'studentRegistration': 22566, 'studentAssessment': 173}


In [3]:
courses, assessments, vle = df["courses"], df["assessments"], df["vle"]
student_info, student_reg = df["studentInfo"], df["studentRegistration"]
student_assess, svle = df["studentAssessment"], df["studentVle"]

# Sổ ghi các vấn đề tìm thấy để tự động đưa vào báo cáo
ISSUES = []
def add_issue(problem, table, count, base, action):
    pct = (count / base) if base else np.nan
    ISSUES.append(dict(problem=problem, table=table, count=int(count),
                       base=int(base), pct=pct, action=action))

def count_dups(d, subset=None):
    """Đếm dòng trùng mà không cấp phát mảng cỡ cả bảng (studentVle có hơn 10 triệu dòng, máy 8 GiB RAM).
    Bảng lớn được xử lý từng nhóm (module, đợt): trùng chỉ có thể xảy ra trong cùng nhóm
    vì hai cột này nằm trong khóa, nên kết quả vẫn chính xác."""
    if len(d) < 2_000_000 or "code_module" not in d.columns:
        return int(d.duplicated(subset).sum())
    total = 0
    for _, g in d.groupby(["code_module", "code_presentation"], observed=True, sort=False):
        total += int(g.duplicated(subset).sum())
    return total


## 2. Thống kê từng bảng (số dòng, số cột, kiểu dữ liệu, thiếu, trùng)

In [4]:
rows = []
for name in TABLES:
    d = df[name]
    rows.append(dict(table=name, rows=len(d), cols=d.shape[1],
                     missing_cells=int(d.isna().sum().sum()),
                     duplicate_rows=count_dups(d),
                     ram_mb=round(d.memory_usage(deep=True).sum() / 2**20, 1)))
table_stats = pd.DataFrame(rows).set_index("table")
display(table_stats)
table_stats.to_csv(RESULTS_DIR / "data_quality_table_stats.csv")

for name in TABLES:
    d = df[name]
    prof = pd.DataFrame({
        "dtype": d.dtypes.astype(str),
        "n_missing": d.isna().sum(),
        "pct_missing": d.isna().mean() * 100,
        "n_unique": d.nunique(),
    })
    print(f"\n--- {name} ---")
    display(prof)

,rows,cols,missing_cells,duplicate_rows,ram_mb
table,,,,,
courses,22,3,0,0,0.0000
assessments,206,6,11,0,0.0000
vle,6364,6,10486,0,0.4000
studentInfo,32593,12,1111,0,4.7000
studentRegistration,32593,5,22566,0,1.4000
studentAssessment,173912,5,173,0,5.3000
studentVle,10655280,6,0,787170,142.3000



--- courses ---


,dtype,n_missing,pct_missing,n_unique
code_module,str,0,0.0000,7
code_presentation,str,0,0.0000,4
module_presentation_length,int64,0,0.0000,7



--- assessments ---


,dtype,n_missing,pct_missing,n_unique
code_module,str,0,0.0000,7
code_presentation,str,0,0.0000,4
id_assessment,int64,0,0.0000,206
assessment_type,str,0,0.0000,3
date,float64,11,5.3398,74
weight,float64,0,0.0000,24



--- vle ---


... (5 outputs left out)


--- studentAssessment ---


,dtype,n_missing,pct_missing,n_unique
id_assessment,int32,0,0.0000,188
id_student,int32,0,0.0000,23369
date_submitted,int64,0,0.0000,312
is_banked,int64,0,0.0000,2
score,float64,173,0.0995,101



--- studentVle ---


,dtype,n_missing,pct_missing,n_unique
code_module,category,0,0.0000,7
code_presentation,category,0,0.0000,4
id_student,int32,0,0.0000,26074
id_site,int32,0,0.0000,6268
date,int16,0,0.0000,295
sum_click,int16,0,0.0000,498


## 3. Các điểm đã biết

Mỗi mục ghi số lượng vào sổ `ISSUES` để tự động có trong báo cáo.

### 3.1 `studentVle`: ngày âm, ngoài cửa sổ, ngoại lệ click

In [5]:
n_vle = len(svle)
d = svle["date"]
n_neg = int((d < 0).sum())
n_win = int(((d >= 0) & (d < CUTOFF_DAY)).sum())
n_late = int((d >= CUTOFF_DAY).sum())
print(f"date: min={d.min()}, max={d.max()}")
print(f"ngày âm (trước khi khóa bắt đầu): {n_neg:,} ({n_neg/n_vle:.2%} theo số dòng)")
print(f"trong cửa sổ 0..{CUTOFF_DAY-1}   : {n_win:,} ({n_win/n_vle:.2%} theo số dòng)")
print(f"từ ngày {CUTOFF_DAY} trở đi        : {n_late:,} ({n_late/n_vle:.2%} theo số dòng)")

clk = svle["sum_click"]
n_nonpos = int((clk <= 0).sum())
p999 = float(clk.quantile(0.999))
n_extreme = int((clk > p999).sum())
print(f"sum_click: min={clk.min()}, median={clk.median()}, p99.9={p999:.0f}, max={clk.max()}; <=0: {n_nonpos:,}")

add_issue("Click có ngày âm (hoạt động trước ngày bắt đầu khóa; tỉ lệ theo số dòng)", "studentVle", n_neg, n_vle,
          f"Loại khỏi cửa sổ feature (chỉ giữ 0 <= date < {CUTOFF_DAY}); cần Trường xác nhận.")
add_issue(f"Click từ ngày {CUTOFF_DAY} trở đi (ngoài cửa sổ dự đoán; tỉ lệ theo số dòng)", "studentVle", n_late, n_vle,
          "Không phải lỗi; ETL phải lọc bỏ để tránh rò rỉ thời gian. Dùng tham số cutoff_day.")
add_issue("sum_click <= 0", "studentVle", n_nonpos, n_vle,
          "Kiểm tra thủ công nếu > 0; nếu có thì loại hoặc ghi chú.")
add_issue(f"sum_click cực lớn (> p99.9 = {p999:.0f}, max = {clk.max()})", "studentVle", n_extreme, n_vle,
          "Giữ nguyên; xử lý lệch phải bằng log1p ở EDA (biểu đồ 8), không cắt dữ liệu.")

date: min=-25, max=269
ngày âm (trước khi khóa bắt đầu): 688,988 (6.47% theo số dòng)
trong cửa sổ 0..27   : 2,112,323 (19.82% theo số dòng)
từ ngày 28 trở đi        : 7,853,969 (73.71% theo số dòng)
sum_click: min=1, median=2.0, p99.9=88, max=6977; <=0: 0


### 3.2 `studentRegistration`: `date_registration` và `date_unregistration`

In [6]:
n_reg = len(student_reg)
dr, du = student_reg["date_registration"], student_reg["date_unregistration"]

n_dr_na = int(dr.isna().sum())
n_dr_neg = int((dr < 0).sum())
print(f"date_registration: thiếu={n_dr_na}, âm={n_dr_neg:,} ({n_dr_neg/n_reg:.2%}), min={dr.min()}, max={dr.max()}")

n_du_na = int(du.isna().sum())
n_du_has = int(du.notna().sum())
n_du_neg = int((du < 0).sum())
n_du_early = int((du < CUTOFF_DAY).sum())      
n_du_late = int((du >= CUTOFF_DAY).sum())
print(f"date_unregistration: trống={n_du_na:,} ({n_du_na/n_reg:.2%}); có giá trị={n_du_has:,}")
print(f"  rút trước ngày {CUTOFF_DAY} (ngoài cohort): {n_du_early:,} ({n_du_early/n_reg:.2%}); trong đó âm: {n_du_neg}")
print(f"  rút từ ngày {CUTOFF_DAY} (vẫn trong cohort): {n_du_late:,}")

n_cohort = n_reg - n_du_early
print(f"Cỡ cohort (còn đăng ký ở đầu ngày {CUTOFF_DAY}) = {n_cohort:,} / {n_reg:,}")

add_issue("date_registration thiếu", "studentRegistration", n_dr_na, n_reg,
          "Điền median TÍNH TRÊN TRAIN + cờ date_registration_missing (làm ở pipeline của Trường).")
add_issue("date_registration âm", "studentRegistration", n_dr_neg, n_reg,
          "Hợp lệ (đăng ký trước ngày bắt đầu); giữ nguyên làm feature số.")
add_issue("date_unregistration trống (không rút lui)", "studentRegistration", n_du_na, n_reg,
          "Không phải lỗi. KHÔNG dùng date_unregistration làm feature; chỉ dùng để xác định cohort.")
add_issue(f"Rút lui trước ngày {CUTOFF_DAY} (ngoài cohort)", "studentRegistration", n_du_early, n_reg,
          "Loại khỏi cohort, lưu data/processed/excluded_early_withdrawn.csv và thống kê riêng.")
add_issue("date_unregistration âm (rút trước ngày bắt đầu)", "studentRegistration", n_du_neg, n_reg,
          "Thuộc nhóm ngoài cohort ở trên; ghi nhận để giải thích biểu đồ rút lui theo ngày.")

date_registration: thiếu=45, âm=32,312 (99.14%), min=-322.0, max=167.0
date_unregistration: trống=22,521 (69.10%); có giá trị=10,072
  rút trước ngày 28 (ngoài cohort): 5,040 (15.46%); trong đó âm: 2678
  rút từ ngày 28 (vẫn trong cohort): 5,032
Cỡ cohort (còn đăng ký ở đầu ngày 28) = 27,553 / 32,593


In [7]:
reg_cohort = student_reg.assign(in_cohort=~(student_reg["date_unregistration"] < CUTOFF_DAY))
cohort_by_mp = (reg_cohort.groupby(["code_module", "code_presentation"])
                .agg(registered=("id_student", "size"), in_cohort=("in_cohort", "sum"))
                .assign(excluded=lambda x: x.registered - x.in_cohort,
                        pct_excluded=lambda x: x.excluded / x.registered * 100))
display(cohort_by_mp)
cohort_by_mp.to_csv(RESULTS_DIR / "data_quality_cohort_by_module.csv")

registered  in_cohort  excluded  pct_excluded
code_module code_presentation                                               
AAA         2013J                     383        370        13        3.3943
            2014J                     365        348        17        4.6575
BBB         2013B                    1767       1531       236       13.3560
            2013J                    2237       1835       402       17.9705
            2014B                    1613       1282       331       20.5208
            2014J                    2292       1787       505       22.0332
CCC         2014B                    1936       1500       436       22.5207
            2014J                    2498       2023       475       19.0152
DDD         2013B                    1303       1146       157       12.0491
            2013J                    1938       1648       290       14.9639
            2014B                    1228       1030       198       16.1238
            2014J                    1803       1463       340       18.8575
EEE         2013J                    1052        927       125       11.8821
            2014B                     694        605        89       12.8242
            2014J                    1188       1021       167       14.0572
FFF         2013B                    1614       1454       160        9.9133
            2013J                    2283       1983       300       13.1406
            2014B                    1500       1302       198       13.2000
            2014J                    2365       1860       505       21.3531
GGG         2013J                     952        924        28        2.9412
            2014B                     833        797        36        4.3217
            2014J                     749        717        32        4.2724

### 3.3 `studentInfo`: giá trị thiếu và bộ giá trị của cột phân loại

In [8]:
n_info = len(student_info)
n_imd_na = int(student_info["imd_band"].isna().sum())
print(f"imd_band thiếu: {n_imd_na:,} ({n_imd_na/n_info:.2%})")
print(f"số id_student khác nhau: {student_info['id_student'].nunique():,} / {n_info:,} dòng "
      "(một sinh viên có thể học nhiều module/đợt, nên đơn vị phân tích là lượt đăng ký)")

for c in ["code_module", "code_presentation", "gender", "region", "highest_education",
          "imd_band", "age_band", "disability", "final_result"]:
    if c == "final_result":
        print("\nfinal_result: có trong bảng (CHỈ để kiểm tra tên giá trị, không dùng làm feature):",
              sorted(student_info[c].dropna().unique()))
        continue
    print(f"\n{c}: ", sorted(student_info[c].dropna().astype(str).unique()))

print("\nnum_of_prev_attempts:", student_info["num_of_prev_attempts"].describe().round(2).to_dict())
print("studied_credits     :", student_info["studied_credits"].describe().round(2).to_dict())

add_issue("imd_band thiếu", "studentInfo", n_imd_na, n_info,
          "Điền mode TÍNH TRÊN TRAIN + cờ imd_missing (pipeline của Trường).")
imd_vals = student_info["imd_band"].dropna().astype(str).unique()
n_imd_nopct = int(student_info["imd_band"].dropna().astype(str).str.endswith("%").eq(False).sum())
add_issue("imd_band: nhãn dải không đồng nhất (một số không có dấu '%')", "studentInfo", n_imd_nopct, n_info,
          "Giữ nguyên nhãn gốc (kể cả '10-20' không có %), không đổi sang số; Trường mã hóa theo config/columns.yaml trong pipeline.py.")

imd_band thiếu: 1,111 (3.41%)
số id_student khác nhau: 28,785 / 32,593 dòng (một sinh viên có thể học nhiều module/đợt, nên đơn vị phân tích là lượt đăng ký)

code_module:  ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']

code_presentation:  ['2013B', '2013J', '2014B', '2014J']

gender:  ['F', 'M']

region:  ['East Anglian Region', 'East Midlands Region', 'Ireland', 'London Region', 'North Region', 'North Western Region', 'Scotland', 'South East Region', 'South Region', 'South West Region', 'Wales', 'West Midlands Region', 'Yorkshire Region']

highest_education:  ['A Level or Equivalent', 'HE Qualification', 'Lower Than A Level', 'No Formal quals', 'Post Graduate Qualification']

imd_band:  ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']

age_band:  ['0-35', '35-55', '55<=']

disability:  ['N', 'Y']

final_result: có trong bảng (CHỈ để kiểm tra tên giá trị, không dùng làm feature): ['Distinction', 'Fail', 'Pass', 'Withdrawn']

num_o

### 3.4 `assessments` và `studentAssessment`: Exam không ngày, `is_banked`, điểm

In [9]:
n_ass = len(assessments)
exam_no_date = int((assessments["assessment_type"].eq("Exam") & assessments["date"].isna()).sum())
other_no_date = int((~assessments["assessment_type"].eq("Exam") & assessments["date"].isna()).sum())
print("Số bài theo loại:", assessments["assessment_type"].value_counts().to_dict())
print(f"Bài không có ngày: Exam={exam_no_date}, loại khác={other_no_date}")
add_issue("Bài Exam không có ngày", "assessments", exam_no_date, n_ass,
          "Bỏ toàn bộ bài Exam khỏi feature (ngoài cửa sổ dự đoán).")
if other_no_date:
    add_issue("Bài không phải Exam nhưng thiếu ngày hạn", "assessments", other_no_date, n_ass,
              "Báo Trường: không xác định được bài có nằm trước ngày 28 không.")

n_sa = len(student_assess)
n_banked = int((student_assess["is_banked"] == 1).sum())
print(f"is_banked = 1: {n_banked:,} ({n_banked/n_sa:.2%})")
add_issue("Bài nộp is_banked = 1 (điểm chuyển từ lần học trước)", "studentAssessment", n_banked, n_sa,
          "Bỏ khỏi feature (Trường đã chốt).")

sc = student_assess["score"]
n_sc_na = int(sc.isna().sum())
n_sc_out = int(((sc < 0) | (sc > 100)).sum())
print(f"score thiếu={n_sc_na:,}; ngoài [0,100]={n_sc_out}; min={sc.min()}, max={sc.max()}")
add_issue("score thiếu", "studentAssessment", n_sc_na, n_sa,
          "ETL hiện điền score thiếu = 0 và vẫn tính vào avg_score (coi như 0 điểm, kéo điểm trung bình xuống). Đề xuất: bỏ bài thiếu điểm khỏi avg_score và weighted_score (tính trên bài có điểm); nếu không còn bài nào thì avg_score = 0 + no_assess_flag. Cần Trường xác nhận trước khi sửa ETL.")
add_issue("score ngoài khoảng [0, 100]", "studentAssessment", n_sc_out, n_sa,
          "Kiểm tra thủ công nếu > 0.")

ds = student_assess["date_submitted"]
n_ds_neg = int((ds < 0).sum())
n_ds_win = int((ds < CUTOFF_DAY).sum())
print(f"date_submitted: min={ds.min()}, max={ds.max()}, âm={n_ds_neg}, trước ngày {CUTOFF_DAY}={n_ds_win:,}")
add_issue("date_submitted âm (nộp trước ngày bắt đầu)", "studentAssessment", n_ds_neg, n_sa,
          "Hợp lệ (nộp trước ngày bắt đầu); vẫn tính là nộp trước ngày 28; báo Trường để thống nhất.")

dup_pairs = int(student_assess.duplicated(["id_assessment", "id_student"]).sum())
print(f"Cặp (id_assessment, id_student) trùng: {dup_pairs}")
add_issue("Cặp (id_assessment, id_student) trùng", "studentAssessment", dup_pairs, n_sa,
          "Nếu > 0: giữ lần nộp đầu tiên hoặc báo Trường quyết định.")

Số bài theo loại: {'TMA': 106, 'CMA': 76, 'Exam': 24}
Bài không có ngày: Exam=11, loại khác=0
is_banked = 1: 1,909 (1.10%)
score thiếu=173; ngoài [0,100]=0; min=0.0, max=100.0
date_submitted: min=-11, max=608, âm=2057, trước ngày 28=25,348
Cặp (id_assessment, id_student) trùng: 0


### 3.5 `vle`, `courses`, dòng trùng

In [10]:
n_vle_sites = len(vle)
wf_na = int(vle["week_from"].isna().sum())
wt_na = int(vle["week_to"].isna().sum())
print("activity_type:", vle["activity_type"].value_counts().to_dict())
print(f"week_from thiếu={wf_na:,}, week_to thiếu={wt_na:,} / {n_vle_sites:,}")
print(f"id_site duy nhất trong vle: {vle['id_site'].is_unique}")
add_issue("vle.week_from / week_to thiếu", "vle", wf_na, n_vle_sites,
          "Không dùng hai cột này làm feature; không cần xử lý.")

print("\ncourses:")
display(courses)

for name in TABLES:
    n_dup = count_dups(df[name])
    if n_dup and name == "studentVle":
        n_key_dup = count_dups(svle, KEYS + ["id_site", "date"])
        print(f"studentVle: trùng toàn dòng={n_dup:,}; trùng theo (lượt đăng ký, id_site, date)={n_key_dup:,} "
              f"(trong đó cùng khóa nhưng khác sum_click={n_key_dup - n_dup:,})")
        add_issue("Dòng trùng hoàn toàn (cùng sinh viên, trang, ngày, sum_click)", name, n_dup, len(svle),
                  "CHƯA khử trùng: bảng không có giờ nên đây có thể là nhiều phiên hợp lệ trong cùng một ngày; "
                  "khử trùng sẽ làm tổng click bị thiếu. Trường quyết định; mặc định giữ nguyên và cộng dồn.")
        add_issue("studentVle: nhiều dòng cùng (lượt đăng ký, id_site, date)", name, n_key_dup, len(svle),
                  "Cộng sum_click theo (lượt đăng ký, id_site, date) trước khi tổng hợp feature.")
    elif n_dup:
        add_issue("Dòng trùng hoàn toàn", name, n_dup, len(df[name]),
                  "Khử trùng trước khi tổng hợp; báo Trường.")
dup_summary = {n: count_dups(df[n]) for n in TABLES}
print("\nDòng trùng hoàn toàn theo bảng:", dup_summary)

activity_type: {'resource': 2660, 'subpage': 1055, 'oucontent': 996, 'url': 886, 'forumng': 194, 'quiz': 127, 'page': 102, 'oucollaborate': 82, 'questionnaire': 61, 'ouwiki': 49, 'dataplus': 28, 'externalquiz': 26, 'homepage': 22, 'glossary': 21, 'ouelluminate': 21, 'dualpane': 20, 'repeatactivity': 5, 'htmlactivity': 4, 'sharedsubpage': 3, 'folder': 2}
week_from thiếu=5,243, week_to thiếu=5,243 / 6,364
id_site duy nhất trong vle: True

courses:


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240
5,BBB,2014B,234
6,CCC,2014J,269
7,CCC,2014B,241
8,DDD,2013J,261
9,DDD,2014J,262


studentVle: trùng toàn dòng=787,170; trùng theo (lượt đăng ký, id_site, date)=2,195,960 (trong đó cùng khóa nhưng khác sum_click=1,408,790)

Dòng trùng hoàn toàn theo bảng: {'courses': 0, 'assessments': 0, 'vle': 0, 'studentInfo': 0, 'studentRegistration': 0, 'studentAssessment': 0, 'studentVle': 787170}


## 4. Kiểm tra khóa và ghép bảng

Đơn vị phân tích là lượt đăng ký `(id_student, code_module, code_presentation)`. Ghép thiếu khóa (chỉ `id_student`) sẽ nhân dòng vì một sinh viên học nhiều module/đợt.

In [11]:
dup_info = int(student_info.duplicated(KEYS).sum())
dup_reg = int(student_reg.duplicated(KEYS).sum())
print(f"Khóa trùng: studentInfo={dup_info}, studentRegistration={dup_reg}")
add_issue("Khóa (id_student, code_module, code_presentation) trùng", "studentInfo", dup_info, len(student_info),
          "Phải = 0 để ghép an toàn; nếu > 0 báo Trường.")
add_issue("Khóa (id_student, code_module, code_presentation) trùng", "studentRegistration", dup_reg, len(student_reg),
          "Phải = 0 để ghép an toàn; nếu > 0 báo Trường.")

m3 = student_info.merge(student_reg, on=KEYS, how="outer", indicator=True)
print("\nGhép studentInfo × studentRegistration bằng 3 khóa:")
print("  số dòng sau ghép:", len(m3), "| studentInfo:", len(student_info), "| studentRegistration:", len(student_reg))
print("  ", m3["_merge"].value_counts().to_dict())
left_only = int((m3["_merge"] == "left_only").sum())
right_only = int((m3["_merge"] == "right_only").sum())
add_issue("Lượt đăng ký có trong studentInfo nhưng không có trong studentRegistration", "studentInfo",
          left_only, len(student_info), "Nếu > 0: date_registration sẽ thiếu cho các dòng này; báo Trường.")
add_issue("Lượt đăng ký có trong studentRegistration nhưng không có trong studentInfo", "studentRegistration",
          right_only, len(student_reg), "Nếu > 0: không có nhân khẩu học; báo Trường để loại.")

cols_reg = KEYS + ["date_registration"]
inflated_1 = len(student_info.merge(student_reg[cols_reg], on="id_student", how="left", suffixes=("", "_r")))
inflated_2 = len(student_info.merge(student_reg[cols_reg], on=["id_student", "code_module"], how="left", suffixes=("", "_r")))
print(f"\nGhép chỉ id_student                  : {inflated_1:,} dòng (gốc {len(student_info):,})")
print(f"Ghép id_student + code_module        : {inflated_2:,} dòng")
print(f"Ghép đủ 3 khóa (left, validate 1-1)  : {len(student_info.merge(student_reg[cols_reg], on=KEYS, how='left', validate='one_to_one')):,} dòng")

Khóa trùng: studentInfo=0, studentRegistration=0

Ghép studentInfo × studentRegistration bằng 3 khóa:
  số dòng sau ghép: 32593 | studentInfo: 32593 | studentRegistration: 32593
   {'both': 32593, 'left_only': 0, 'right_only': 0}

Ghép chỉ id_student                  : 40,801 dòng (gốc 32,593)
Ghép id_student + code_module        : 35,291 dòng
Ghép đủ 3 khóa (left, validate 1-1)  : 32,593 dòng


In [12]:
info_keys = student_info[KEYS].drop_duplicates()

vle_keys = svle.groupby(KEYS, observed=True, sort=False).size().reset_index()[KEYS]   # nhẹ RAM hơn drop_duplicates trên 10 triệu dòng
orph_vle = vle_keys.merge(info_keys, on=KEYS, how="left", indicator=True)
n_orph_vle = int((orph_vle["_merge"] == "left_only").sum())
print(f"Bộ khóa trong studentVle: {len(vle_keys):,}; không có trong studentInfo: {n_orph_vle}")
add_issue("Bộ khóa lượt đăng ký trong studentVle không có trong studentInfo", "studentVle",
          n_orph_vle, len(vle_keys), "Nếu > 0: loại các dòng đó khi ghép feature.")

n_site_orph = int((~svle["id_site"].isin(vle["id_site"])).sum())
print(f"studentVle.id_site không có trong vle: {n_site_orph:,}")
add_issue("studentVle.id_site không có trong bảng vle", "studentVle", n_site_orph, len(svle),
          "Nếu > 0: không gán được activity_type; giữ dòng cho tổng click, báo Trường.")

sa = student_assess.merge(assessments[["id_assessment", "code_module", "code_presentation"]],
                          on="id_assessment", how="left", validate="many_to_one")
n_sa_orph_ass = int(sa["code_module"].isna().sum())
sa_keys = sa.dropna(subset=["code_module"])[KEYS].drop_duplicates()
orph_sa = sa_keys.merge(info_keys, on=KEYS, how="left", indicator=True)
n_orph_sa = int((orph_sa["_merge"] == "left_only").sum())
print(f"studentAssessment: id_assessment không có trong assessments = {n_sa_orph_ass}; "
      f"bộ khóa không có trong studentInfo = {n_orph_sa}")
add_issue("studentAssessment.id_assessment không có trong assessments", "studentAssessment",
          n_sa_orph_ass, len(student_assess), "Nếu > 0: không xác định được module/đợt; loại.")
add_issue("Bộ khóa lượt đăng ký trong studentAssessment không có trong studentInfo", "studentAssessment",
          n_orph_sa, len(sa_keys), "Nếu > 0: loại khi ghép feature.")

agg = svle.groupby(KEYS, observed=True).size().rename("n_rows").reset_index()
agg["code_module"] = agg["code_module"].astype(str)
agg["code_presentation"] = agg["code_presentation"].astype(str)
chk = student_info[KEYS].astype({"code_module": str, "code_presentation": str}).merge(
    agg, on=KEYS, how="left", validate="one_to_one")
print(f"\nGhép studentInfo ← studentVle (đã gộp theo 3 khóa): {len(chk):,} dòng (gốc {len(student_info):,}); "
      f"không có hoạt động VLE: {int(chk['n_rows'].isna().sum()):,} lượt đăng ký")
no_vle = int(chk["n_rows"].isna().sum())
add_issue("Lượt đăng ký không có bản ghi VLE nào", "studentInfo", no_vle, len(student_info),
          "Biến đếm = 0 (quy tắc giá trị thiếu đã chốt), không bỏ dòng.")

Bộ khóa trong studentVle: 29,228; không có trong studentInfo: 0
studentVle.id_site không có trong vle: 0
studentAssessment: id_assessment không có trong assessments = 0; bộ khóa không có trong studentInfo = 0

Ghép studentInfo ← studentVle (đã gộp theo 3 khóa): 32,593 dòng (gốc 32,593); không có hoạt động VLE: 3,365 lượt đăng ký


## 5. Feature nhân khẩu học (chạy thử `src/etl/features_demographic.py`)

Hàm **chỉ mã hóa**: giữ nguyên chuỗi gốc của `gender`, `region`, `highest_education`, `imd_band`, `age_band`, `disability` (chỉ bỏ khoảng trắng thừa); giữ `NaN` ở `imd_band` và `date_registration` cùng hai cờ thiếu. Việc đổi chữ thành số, one-hot và điền mode/median do pipeline của Trường làm (theo `config/columns.yaml`, fit trên train).

In [13]:
from src.etl.features_demographic import (
    build_demographic_features, check_demographic_table, feature_columns, FORBIDDEN_COLUMNS)

demo = build_demographic_features(student_info, student_reg)
errors = check_demographic_table(demo, expected_rows=len(student_info))
print("Kích thước:", demo.shape, "| số cột feature:", len(feature_columns()))
print("Kiểm tra:", "ĐẠT" if not errors else errors)
assert not errors, errors
assert not set(FORBIDDEN_COLUMNS) & set(demo.columns)

display(demo.head())
na = demo.isna().sum()
print("Giá trị thiếu còn lại:")
display(na[na > 0].to_frame("n_missing"))
print("Khớp cờ:", int(demo['imd_missing'].sum()) == n_imd_na, int(demo['date_registration_missing'].sum()) == n_dr_na)

for c in ["highest_education", "imd_band", "age_band", "disability"]:  # nhãn chuỗi gốc, không phải số
    print(f"\n{c}:", demo[c].value_counts(dropna=False).sort_index().to_dict())

Kích thước: (32593, 14) | số cột feature: 11
Kiểm tra: ĐẠT


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,imd_missing,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,date_registration_missing
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,0,55<=,0,240,N,-159.0000,0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,0,35-55,0,60,N,-53.0000,0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,0,35-55,0,60,Y,-92.0000,0
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,0,35-55,0,60,N,-52.0000,0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0,0-35,0,60,N,-176.0000,0


Giá trị thiếu còn lại:


,n_missing
imd_band,1111
date_registration,45


Khớp cờ: True True

highest_education: {'A Level or Equivalent': 14045, 'HE Qualification': 4730, 'Lower Than A Level': 13158, 'No Formal quals': 347, 'Post Graduate Qualification': 313}

imd_band: {'0-10%': 3311, '10-20': 3516, '20-30%': 3654, '30-40%': 3539, '40-50%': 3256, '50-60%': 3124, '60-70%': 2905, '70-80%': 2879, '80-90%': 2762, '90-100%': 2536, nan: 1111}

age_band: {'0-35': 22944, '35-55': 9433, '55<=': 216}

disability: {'N': 29429, 'Y': 3164}


## 6. Xuất báo cáo `results/data_quality_report.md`

Báo cáo gồm bảng thống kê 7 bảng, bảng vấn đề (số lượng, tỉ lệ, đề xuất xử lý), cỡ cohort và các điểm cần xác nhận.

In [14]:
import datetime as _dt

issues_df = pd.DataFrame(ISSUES)
issues_df["pct_str"] = issues_df["pct"].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
issues_df.to_csv(RESULTS_DIR / "data_quality_issues.csv", index=False)

def md_table(frame):
    cols = list(frame.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in frame.iterrows():
        out.append("| " + " | ".join(str(r[c]).replace("|", "/") for c in cols) + " |")
    return "\n".join(out)

stats_md = table_stats.reset_index().rename(columns={
    "table": "Bảng", "rows": "Số dòng", "cols": "Số cột",
    "missing_cells": "Ô thiếu", "duplicate_rows": "Dòng trùng", "ram_mb": "RAM (MB)"})
stats_md["Số dòng"] = stats_md["Số dòng"].map("{:,}".format)
stats_md["Ô thiếu"] = stats_md["Ô thiếu"].map("{:,}".format)

# chỉ liệt kê vấn đề thật sự có (count > 0); mục count = 0 gom thành một dòng "đã kiểm tra, không có"
found = issues_df[issues_df["count"] > 0]
clean = issues_df[issues_df["count"] == 0]
found_md = found.assign(count=found["count"].map("{:,}".format))[
    ["problem", "table", "count", "pct_str", "action"]].rename(columns={
    "problem": "Vấn đề", "table": "Bảng", "count": "Số lượng", "pct_str": "Tỉ lệ", "action": "Đề xuất xử lý"})

report = f'''# Báo cáo chất lượng dữ liệu OULAD

## 1. Tổng quan 7 bảng

{md_table(stats_md)}

Số ô '?' thô theo bảng: {('; '.join(f'{k}: {v:,}' for k, v in qmarks.items() if v) or 'không có (thiếu được ghi bằng ô rỗng)')}.

## 2. Cohort

- Tổng lượt đăng ký (studentRegistration): {n_reg:,}.
- Rút lui trước ngày {CUTOFF_DAY} (ngoài cohort): {n_du_early:,} ({n_du_early/n_reg:.2%}); cohort còn lại: {n_cohort:,}.

## 3. Vấn đề tìm thấy và đề xuất xử lý

{md_table(found_md)}

Đã kiểm tra và không có ({clean['problem'].nunique()} mục): {('; '.join(clean['problem'].drop_duplicates()) or 'không có mục nào')}.

## 4. Kiểm tra khóa

- Đơn vị phân tích: lượt đăng ký `(id_student, code_module, code_presentation)`; {student_info['id_student'].nunique():,} sinh viên khác nhau trên {len(student_info):,} lượt đăng ký.
- Ghép studentInfo với studentRegistration bằng đủ 3 khóa: {len(student_info.merge(student_reg[cols_reg], on=KEYS, how='left', validate='one_to_one')):,} dòng (không tăng so với {len(student_info):,}).
- Ghép chỉ bằng `id_student`: {inflated_1:,} dòng, tức tăng ngoài ý muốn. Luôn ghép đủ 3 khóa.

## 5. Cần xác nhận

1. Click có `date < 0` trong studentVle: loại khỏi cửa sổ, chỉ giữ 0 <= date < {CUTOFF_DAY} (đã chốt).
2. `imd_band` và `date_registration`: chỉ giữ nguyên giá trị và tạo cờ thiếu; Trường điền mode/median trên train trong `pipeline.py`.
3. One-hot của `code_module`, `gender`, `region` do `pipeline.py` của Trường làm (OneHotEncoder fit trên train, danh sách cột cố định cho cả ba tập); `features_demographic.py` không tự one-hot. Hàm cũng không đổi chữ thành số: `gender`, `region`, `highest_education`, `imd_band`, `age_band`, `disability` giữ nguyên chuỗi gốc (`imd_band` giữ cả nhãn "10-20" không có %), pipeline mã hóa theo `config/columns.yaml`.
4. Bài thiếu `score` ({n_sc_na:,} dòng, {n_sc_na/n_sa:.2%} bài nộp): ETL hiện điền 0 và vẫn tính vào `avg_score`. Đề xuất của Đạt: bỏ bài thiếu điểm khỏi `avg_score` và `weighted_score`; nếu không còn bài nào thì `avg_score` = 0 kèm `no_assess_flag`. Cần Trường chọn: bỏ bài thiếu điểm (đề xuất) hay giữ điền 0.
5. Dòng trùng hoàn toàn trong studentVle: giữ và cộng dồn `sum_click`, hay khử trùng?

## 6. Feature nhân khẩu học

`src/etl/features_demographic.py` đã chạy thử trên dữ liệu thật: {demo.shape[0]:,} dòng, {len(feature_columns())} cột feature (11 cột, không one-hot, không đổi chữ thành số), đạt `check_demographic_table` (nhãn nằm trong danh sách nhãn gốc).
'''
(RESULTS_DIR / "data_quality_report.md").write_text(report, encoding="utf-8")

6183